**Imports**

In [1]:
import pandas as pd
import numpy as np
import os
import joblib
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
 
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.preprocessing import LabelEncoder
 
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
 
os.makedirs('../data/splits',          exist_ok=True)
os.makedirs('../models/saved_models',  exist_ok=True)
os.makedirs('../models/embeddings',    exist_ok=True)
os.makedirs('../results',              exist_ok=True)
 
print("Imports done.")

Imports done.


**Load your cleaned dataset**

In [2]:
DATA_PATH = '../data/animal_disease_merged_cleaned.csv'
df = pd.read_csv(DATA_PATH)
 
print(f"Dataset: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Species:  {df['Species'].nunique()}")
print(f"Diseases: {df['Disease_Prediction'].nunique()}")
print(f"Missing:  {df.isnull().sum().sum()}")
 
counts = df['Disease_Prediction'].value_counts()
print(f"\nImbalance ratio: {counts.max()}:{counts.min()}")
print(f"Classes with 1 sample: {(counts==1).sum()}")

Dataset: 59,315 rows × 30 columns
Species:  16
Diseases: 162
Missing:  0

Imbalance ratio: 8941:1
Classes with 1 sample: 41


**Build symptom text strings**

In [3]:
def build_symptom_text(row):
    symptom_list = []
    for col in ['Symptom_1', 'Symptom_2', 'Symptom_3', 'Symptom_4']:
        val = str(row.get(col, '')).strip()
        if val and val.lower() not in ['nan', 'none', 'not_recorded', '']:
            symptom_list.append(val)
 
    primary    = symptom_list[0] if symptom_list else 'unknown'
    additional = ', '.join(symptom_list[1:]) if len(symptom_list) > 1 else 'none'
 
    binary_map = {
        'Appetite_Loss': 'appetite loss', 'Vomiting': 'vomiting',
        'Diarrhea': 'diarrhea', 'Coughing': 'coughing',
        'Labored_Breathing': 'labored breathing', 'Lameness': 'lameness',
        'Skin_Lesions': 'skin lesions', 'Nasal_Discharge': 'nasal discharge',
        'Eye_Discharge': 'eye discharge',
    }
    present = [lbl for col, lbl in binary_map.items() if row.get(col, 0) == 1]
    absent  = [lbl for col, lbl in binary_map.items() if row.get(col, 0) == 0]
 
    return (
        f"Species: {row.get('Species','unknown')}. "
        f"Breed: {row.get('Breed','unknown')}. "
        f"Age: {row.get('Age','unknown')} years. "
        f"Gender: {row.get('Gender','unknown')}. "
        f"Weight: {row.get('Weight','unknown')} kg. "
        f"Primary symptom: {primary}. "
        f"Additional symptoms: {additional}. "
        f"Duration: {row.get('Duration','unknown')}. "
        f"Symptoms present: {', '.join(present) if present else 'none'}. "
        f"Symptoms absent: {', '.join(absent) if absent else 'none'}. "
        f"Body temperature: {row.get('Body_Temperature','unknown')} C. "
        f"Heart rate: {row.get('Heart_Rate','unknown')} bpm. "
        f"Danger level: {row.get('Danger_Level_Category','unknown')}."
    )
 
print("Building symptom text strings...")
df['symptom_text'] = df.apply(build_symptom_text, axis=1)
 
print(f"\nExample text:")
print(df['symptom_text'].iloc[0])

Building symptom text strings...

Example text:
Species: Cow. Breed: Holstein. Age: 3 years. Gender: Female. Weight: 600.0 kg. Primary symptom: Fever. Additional symptoms: Nasal Discharge, Labored Breathing, Coughing. Duration: 5 days. Symptoms present: appetite loss, coughing, labored breathing, nasal discharge. Symptoms absent: vomiting, diarrhea, lameness, skin lesions, eye discharge. Body temperature: 40.1 C. Heart rate: 90 bpm. Danger level: High.


**Build symptom text strings**

In [4]:
le = LabelEncoder()
df['label'] = le.fit_transform(df['Disease_Prediction'])
 
print(f"\nLabel encoding: {len(le.classes_)} classes")
joblib.dump(le, '../models/saved_models/label_encoder.pkl')
print("Saved: models/saved_models/label_encoder.pkl")


Label encoding: 162 classes
Saved: models/saved_models/label_encoder.pkl


**Stratified split 70/15/15**

In [5]:
# Handles singleton classes (only 1 sample — cannot be stratified)
STRUCTURED_COLS = [
    'Age', 'Weight', 'Appetite_Loss', 'Vomiting', 'Diarrhea',
    'Coughing', 'Labored_Breathing', 'Lameness', 'Skin_Lesions',
    'Nasal_Discharge', 'Eye_Discharge', 'Body_Temperature',
    'Heart_Rate', 'Symptom_Count', 'Temp_Risk', 'Symptom_Risk',
    'Critical_Risk', 'Dangerous_Level'
]
 
feature_cols = ['symptom_text'] + STRUCTURED_COLS
X = df[feature_cols]
y = df['Disease_Prediction']
 
# Separate singletons — StratifiedShuffleSplit needs at least 2 per class
counts_label  = y.value_counts()
single_mask   = y.isin(counts_label[counts_label == 1].index)
X_single      = X[single_mask]
y_single      = y[single_mask]
X_split       = X[~single_mask]
y_split       = y[~single_mask]
 
# 85% train+val, 15% test
sss1 = StratifiedShuffleSplit(n_splits=1, test_size=0.15, random_state=RANDOM_STATE)
for tv_idx, test_idx in sss1.split(X_split, y_split):
    X_tv, X_test = X_split.iloc[tv_idx], X_split.iloc[test_idx]
    y_tv, y_test = y_split.iloc[tv_idx], y_split.iloc[test_idx]
 
# From 85%, take 17.6% as val → 15% of total
sss2 = StratifiedShuffleSplit(n_splits=1, test_size=0.176, random_state=RANDOM_STATE)
for train_idx, val_idx in sss2.split(X_tv, y_tv):
    X_train, X_val = X_tv.iloc[train_idx], X_tv.iloc[val_idx]
    y_train, y_val = y_tv.iloc[train_idx], y_tv.iloc[val_idx]
 
# Add singletons to training set
X_train = pd.concat([X_train, X_single], ignore_index=True)
y_train = pd.concat([y_train, y_single], ignore_index=True)
 
print(f"\nSplit complete:")
print(f"  Train: {len(X_train):,} ({len(X_train)/len(df)*100:.1f}%)")
print(f"  Val:   {len(X_val):,}  ({len(X_val)/len(df)*100:.1f}%)")
print(f"  Test:  {len(X_test):,}  ({len(X_test)/len(df)*100:.1f}%)")
print(f"\n  Diseases in train: {y_train.nunique()}")
print(f"  Diseases in val:   {y_val.nunique()}")
print(f"  Diseases in test:  {y_test.nunique()}")


Split complete:
  Train: 41,555 (70.1%)
  Val:   8,868  (15.0%)
  Test:  8,892  (15.0%)

  Diseases in train: 162
  Diseases in val:   95
  Diseases in test:  95


**Encode Species and Breed as ordinal features and save the encoder**

In [6]:
from sklearn.preprocessing import OrdinalEncoder
import joblib
 
# Fit ordinal encoder on training set only
species_breed_cols = ['Species', 'Breed', 'Gender']
 
# Load original dataset to get Species/Breed (not in splits yet — add them)
df_full = pd.read_csv('../data/animal_disease_merged_cleaned.csv')
 
# Re-align with splits using index
X_train_full = pd.read_csv('../data/splits/X_train.csv')
X_val_full   = pd.read_csv('../data/splits/X_val.csv')
X_test_full  = pd.read_csv('../data/splits/X_test.csv')
 
# Save Species, Breed, Gender alongside splits
# (these need to come from the original df, matched by position)
# The splits were created from df in order, so we reconstruct by merging on symptom_text
 
# Simplest approach: re-add Species/Breed by matching symptom_text to original df
# Build a lookup from symptom_text -> Species, Breed, Gender
df_full['symptom_text'] = df_full.apply(build_symptom_text, axis=1)
lookup = df_full.set_index('symptom_text')[['Species', 'Breed', 'Gender']]
 
for split_name, X_split in [('X_train', X_train_full),
                              ('X_val',   X_val_full),
                              ('X_test',  X_test_full)]:
    merged = X_split[['symptom_text']].join(
        lookup, on='symptom_text', how='left')
    X_split['Species'] = merged['Species'].fillna('Unknown')
    X_split['Breed']   = merged['Breed'].fillna('Unknown')
    X_split['Gender']  = merged['Gender'].fillna('Unknown')
    X_split.to_csv(f'../data/splits/{split_name}.csv', index=False)
    print(f"Updated {split_name}.csv with Species/Breed/Gender")
 
# Fit and save the ordinal encoder on training Species/Breed/Gender
ord_enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
ord_enc.fit(X_train_full[species_breed_cols].fillna('Unknown'))
joblib.dump(ord_enc, '../models/saved_models/species_breed_encoder.pkl')
print("Saved species_breed_encoder.pkl")

Updated X_train.csv with Species/Breed/Gender
Updated X_val.csv with Species/Breed/Gender
Updated X_test.csv with Species/Breed/Gender
Saved species_breed_encoder.pkl


**Save splits to disk**

In [7]:
X_train.to_csv('../data/splits/X_train.csv', index=False)
X_val.to_csv('../data/splits/X_val.csv',     index=False)
X_test.to_csv('../data/splits/X_test.csv',   index=False)
y_train.to_csv('../data/splits/y_train.csv', index=False)
y_val.to_csv('../data/splits/y_val.csv',     index=False)
y_test.to_csv('../data/splits/y_test.csv',   index=False)
 
print("\nAll splits saved to data/splits/")
print("\nNOTEBOOK 01 COMPLETE ✅ — proceed to 02_finetuning.py")


All splits saved to data/splits/

NOTEBOOK 01 COMPLETE ✅ — proceed to 02_finetuning.py
